# Grouped TORGO fine-tuning: Whisper-tiny

In Kaggle, import this notebook, enable Internet, select a T4 GPU and **Run All**. The frozen recording manifest is embedded. No account tokens are needed for the public inputs. The notebook uses one GPU. Allow the final cell to finish before saving the notebook version with outputs.

The run checks source files, performs a disposable one-step smoke test, trains from the pinned base model, selects on ASR validation, and exports pause-calibration predictions. Final-test scoring belongs to the later locked evaluation. Download `iciss_grouped_task2.zip` from the Output panel. Keep the checkpoint directory for same-run recovery if the session is interrupted.

In [ ]:
%pip install -q transformers==4.57.6 jiwer==4.0.0 "soundfile>=0.12" "pyarrow>=19"

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import os
import sys
import subprocess
from importlib.metadata import version, PackageNotFoundError
from pathlib import Path

# Select one device before importing torch. Preserves an existing device restriction.
os.environ["CUDA_VISIBLE_DEVICES"] = os.environ.get("CUDA_VISIBLE_DEVICES", "0").split(",")[0]
os.environ["TOKENIZERS_PARALLELISM"] = "false"
if Path("/kaggle/working").is_dir():
    from packaging.requirements import Requirement
    requirements = ["transformers==4.57.6", "jiwer==4.0.0",
                    "soundfile>=0.12", "pyarrow>=19"]
    missing = []
    for spec in requirements:
        req = Requirement(spec)
        try:
            if version(req.name) not in req.specifier:
                missing.append(spec)
        except PackageNotFoundError:
            missing.append(spec)
    if missing:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

## Frozen split and pinned inputs

The embedded payload is the exact released manifest, compressed for transport. Its checksum and the four audio shard hashes are validated before training.

In [ ]:
audit = {
  "all_multiword_recordings_by_speaker": {
    "F01": {
      "asr_train": 12,
      "asr_validation": 2,
      "final_test": 4,
      "pause_calibration": 2
    },
    "F03": {
      "asr_train": 83,
      "asr_validation": 14,
      "final_test": 28,
      "pause_calibration": 14
    },
    "F04": {
      "asr_train": 61,
      "asr_validation": 10,
      "final_test": 20,
      "pause_calibration": 10
    },
    "FC01": {
      "asr_train": 16,
      "asr_validation": 3,
      "final_test": 5,
      "pause_calibration": 2
    },
    "FC02": {
      "asr_train": 190,
      "asr_validation": 32,
      "final_test": 63,
      "pause_calibration": 31
    },
    "FC03": {
      "asr_train": 157,
      "asr_validation": 26,
      "final_test": 52,
      "pause_calibration": 26
    },
    "M01": {
      "asr_train": 53,
      "asr_validation": 9,
      "final_test": 18,
      "pause_calibration": 9
    },
    "M02": {
      "asr_train": 55,
      "asr_validation": 9,
      "final_test": 19,
      "pause_calibration": 9
    },
    "M03": {
      "asr_train": 57,
      "asr_validation": 10,
      "final_test": 19,
      "pause_calibration": 9
    },
    "M04": {
      "asr_train": 52,
      "asr_validation": 9,
      "final_test": 17,
      "pause_calibration": 8
    },
    "M05": {
      "asr_train": 69,
      "asr_validation": 12,
      "final_test": 23,
      "pause_calibration": 11
    },
    "MC01": {
      "asr_train": 171,
      "asr_validation": 29,
      "final_test": 57,
      "pause_calibration": 28
    },
    "MC02": {
      "asr_train": 102,
      "asr_validation": 17,
      "final_test": 34,
      "pause_calibration": 17
    },
    "MC03": {
      "asr_train": 133,
      "asr_validation": 22,
      "final_test": 44,
      "pause_calibration": 22
    },
    "MC04": {
      "asr_train": 157,
      "asr_validation": 26,
      "final_test": 53,
      "pause_calibration": 26
    }
  },
  "counts": {
    "asr_train": {
      "primary_files": 5447,
      "recordings": 5447,
      "source_files": 9934
    },
    "asr_validation": {
      "primary_files": 912,
      "recordings": 912,
      "source_files": 1648
    },
    "excluded": {
      "primary_files": 0,
      "recordings": 0,
      "source_files": 0
    },
    "final_test": {
      "primary_files": 1815,
      "recordings": 1815,
      "source_files": 3318
    },
    "pause_calibration": {
      "primary_files": 901,
      "recordings": 901,
      "source_files": 1652
    }
  },
  "dataset": "abnerh/TORGO-database",
  "dataset_revision": "c4e0b560a515465e61bb555455e34fc620031fe8",
  "dysarthric_multiword_recordings_by_speaker": {
    "F01": {
      "asr_train": 12,
      "asr_validation": 2,
      "final_test": 4,
      "pause_calibration": 2
    },
    "F03": {
      "asr_train": 83,
      "asr_validation": 14,
      "final_test": 28,
      "pause_calibration": 14
    },
    "F04": {
      "asr_train": 61,
      "asr_validation": 10,
      "final_test": 20,
      "pause_calibration": 10
    },
    "M01": {
      "asr_train": 53,
      "asr_validation": 9,
      "final_test": 18,
      "pause_calibration": 9
    },
    "M02": {
      "asr_train": 55,
      "asr_validation": 9,
      "final_test": 19,
      "pause_calibration": 9
    },
    "M03": {
      "asr_train": 57,
      "asr_validation": 10,
      "final_test": 19,
      "pause_calibration": 9
    },
    "M04": {
      "asr_train": 52,
      "asr_validation": 9,
      "final_test": 17,
      "pause_calibration": 8
    },
    "M05": {
      "asr_train": 69,
      "asr_validation": 12,
      "final_test": 23,
      "pause_calibration": 11
    }
  },
  "final_files_with_exact_prompt_in_training": 1742,
  "final_primary_files": 1815,
  "final_prompt_overlap_fraction": 0.9597796143250689,
  "known_test_speakers": [
    "F01",
    "F03",
    "F04",
    "FC01",
    "FC02",
    "FC03",
    "M01",
    "M02",
    "M03",
    "M04",
    "M05",
    "MC01",
    "MC02",
    "MC03",
    "MC04"
  ],
  "legacy_metadata_sha256": "3d5bb346d51f84fd9d2967798941e42c2d9a6dd515f29f26d5f32daf4edaf195",
  "manifest_sha256": "c89bf6a4d688fe2f79a4b8e09e4d747a35bccea8862866c83836136dd46e13fe",
  "primary_channels": {
    "arrayMic": 1222,
    "headMic": 7853
  },
  "primary_files_by_speaker_group": {
    "asr_train": {
      "dysarthria": 1873,
      "healthy": 3574
    },
    "asr_validation": {
      "dysarthria": 315,
      "healthy": 597
    },
    "final_test": {
      "dysarthria": 624,
      "healthy": 1191
    },
    "pause_calibration": {
      "dysarthria": 309,
      "healthy": 592
    }
  },
  "reasons": {
    "included": 9075,
    "outside_duration_limits": 2,
    "secondary_channel": 7475
  },
  "recording_overlaps": {
    "asr_train / asr_validation": 0,
    "asr_train / final_test": 0,
    "asr_train / pause_calibration": 0,
    "asr_validation / final_test": 0,
    "asr_validation / pause_calibration": 0,
    "pause_calibration / final_test": 0
  },
  "seed": 42,
  "source_sha256": {
    "train-00000-of-00004.parquet": "cbf4d90de65fc584066de5b92c624bfc61f29a646c46f8676c43e99fd809c9df",
    "train-00001-of-00004.parquet": "be0a1568d81f6d3afd696639c90d4bb67743a1c2267e1139963abfde8733b679",
    "train-00002-of-00004.parquet": "5074644bed0519a2c40cc20e6feb7619332cd10cce9a0ab7be6f099af69e0e03",
    "train-00003-of-00004.parquet": "26d8c649a198c084b9b366023cda88f99dc67dc85f50ccc5494675c104a5339c"
  }
}
manifest_b64 = (
    "..."
)

In [ ]:
import base64
import gzip
import hashlib
import io
import itertools
import json
import re
import zipfile

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
from huggingface_hub import snapshot_download

DATA_ID = "abnerh/TORGO-database"
BASE_ID = "openai/whisper-tiny"
BASE_REV = "169d4a4341b33bc18d8881c4b69c2e104e1cc0af"
DATA_REV = audit["dataset_revision"]
SEED = 42
output_root = Path(os.environ.get("ICISS_OUTPUT_ROOT", "/kaggle/working"))
run_dir = output_root / ("iciss_grouped_v2_" + audit["manifest_sha256"][:12])
run_dir.mkdir(parents=True, exist_ok=True)
manifest_bytes = gzip.decompress(base64.b64decode(manifest_b64))
assert hashlib.sha256(manifest_bytes).hexdigest() == audit["manifest_sha256"]
(run_dir / "split_manifest.csv").write_bytes(manifest_bytes)
(run_dir / "split_audit.json").write_text(json.dumps(audit, indent=2))
manifest = pd.read_csv(io.BytesIO(manifest_bytes), dtype={"session": str, "uid": str})
parts = ["asr_train", "asr_validation", "pause_calibration", "final_test"]
assert manifest.audio_name.is_unique and manifest.sample_id.is_unique
assert manifest.groupby("recording").partition.nunique().eq(1).all()
primary = manifest.loc[manifest["primary"]].copy()
assert primary.recording.is_unique
for part in parts:
    assert primary.partition.eq(part).sum() == audit["counts"][part]["primary_files"]
for a, b in itertools.combinations(parts, 2):
    assert set(manifest.loc[manifest.partition.eq(a), "recording"]).isdisjoint(
        manifest.loc[manifest.partition.eq(b), "recording"])
print(primary.groupby(["partition", "speech_status"]).size().unstack())

def pinned_snapshot(repo, revision, kind, patterns=None):
    return Path(snapshot_download(repo, revision=revision, repo_type=kind, allow_patterns=patterns))

data_dir = pinned_snapshot(DATA_ID, DATA_REV, "dataset", ["data/*.parquet"]) / "data"
base_dir = pinned_snapshot(BASE_ID, BASE_REV, "model",
                           ["*.json", "*.txt", "model.safetensors"])
def file_sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(4 * 1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

# Keep only the audio used during adaptation and calibration in memory.
wanted = set(primary.loc[primary.partition.isin(parts[:3]), "audio_name"])
refs = manifest.set_index("audio_name").transcription.to_dict()
wave_bytes = {}
seen_names = set()
for filename, expected in sorted(audit["source_sha256"].items()):
    path = data_dir / filename
    assert file_sha256(path) == expected, filename
    for batch in pq.ParquetFile(path).iter_batches(batch_size=64, columns=["audio", "transcription"]):
        for row in batch.to_pylist():
            name = Path(row["audio"]["path"]).name
            assert name not in seen_names and refs[name] == row["transcription"], name
            seen_names.add(name)
            if name in wanted:
                wave_bytes[name] = row["audio"]["bytes"]
assert seen_names == set(refs) and set(wave_bytes) == wanted
print("Verified source files:", len(seen_names), "| primary recordings loaded:", len(wave_bytes))

## Training recipe and validation scoring

Train the encoder for 2,000 AdamW updates. On one GPU, batch 8 with accumulation 2 gives 16 recordings per update. Checkpoint selection uses the primary corpus WER, including deletions from empty predictions. Equal validation WER retains the earlier checkpoint. The smoke model is disposable.

In [ ]:
import gc
import math
import torch
import jiwer
from scipy.signal import resample_poly
from transformers import WhisperForConditionalGeneration, WhisperProcessor, get_linear_schedule_with_warmup, set_seed

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU accelerator in Kaggle and restart the notebook.")
device = torch.device("cuda:0")
processor = WhisperProcessor.from_pretrained(base_dir, language="english", task="transcribe")
tokenizer = processor.tokenizer
steps, batch_size, accumulation = 2000, 8, 2
eval_every = 200
eval_batch_size = 8
def normalize(text, *, punctuation_to_space=False):
    clean = re.sub(r"[^A-Z0-9'\s]", " " if punctuation_to_space else "", text.upper())
    return " ".join(clean.split())

def get_partition(label):
    return primary.loc[primary.partition.eq(label)].sort_values("audio_name").reset_index(drop=True)
train, val, calibration = [get_partition(p) for p in parts[:3]]
assert all(normalize(ref) for ref in pd.concat([train, val, calibration]).transcription)

def make_model():
    model = WhisperForConditionalGeneration.from_pretrained(base_dir).float()
    # Preserve Whisper's fixed sinusoidal encoder positions and tied decoder weights.
    for name, parameter in model.named_parameters():
        if not name.startswith("model.encoder."):
            parameter.requires_grad_(False)
    assert not model.model.encoder.embed_positions.weight.requires_grad
    model.generation_config.language = "english"
    model.generation_config.task = "transcribe"
    model.generation_config.forced_decoder_ids = None
    model.generation_config.suppress_tokens = []
    model.generation_config.num_beams = 1
    model.generation_config.do_sample = False
    model.generation_config.max_length = 225
    # The base checkpoint also stores legacy generation settings in model.config.
    # Keep the explicit decoding policy in generation_config across save/reload.
    for field in ("max_length", "suppress_tokens", "begin_suppress_tokens"):
        setattr(model.config, field, None)
    assert all(name.startswith("model.encoder.") for name, p in model.named_parameters() if p.requires_grad)
    return model.to(device)

def make_optimizer(model):
    # Whisper encoder matrices receive weight decay. Its biases and LayerNorm scales do not.
    decay, no_decay = [], []
    for module in model.model.encoder.modules():
        for name, parameter in module.named_parameters(recurse=False):
            if parameter.requires_grad:
                group = no_decay if name == "bias" or isinstance(module, torch.nn.LayerNorm) else decay
                group.append(parameter)
    return torch.optim.AdamW([{"params": decay, "weight_decay": 0.01},
                              {"params": no_decay, "weight_decay": 0.0}], lr=5e-6)

def features(rows):
    waves = []
    for name in rows.audio_name:
        wav, sr = sf.read(io.BytesIO(wave_bytes[name]), dtype="float32", always_2d=True)
        wav = wav.mean(axis=1)
        if sr != 16000:
            factor = math.gcd(sr, 16000)
            wav = resample_poly(wav, 16000 // factor, sr // factor).astype("float32")
        waves.append(wav)
    return processor.feature_extractor(waves, sampling_rate=16000, return_tensors="pt",
                                       return_attention_mask=True).to(device)

def loss_on(model, rows):
    labels = tokenizer(rows.transcription.tolist(), padding=True, return_tensors="pt")
    ids = labels.input_ids.masked_fill(labels.attention_mask.eq(0), -100)
    assert ids[:, 0].eq(model.config.decoder_start_token_id).all()
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        return model(**features(rows), labels=ids[:, 1:].to(device), use_cache=False).loss

@torch.inference_mode()
def predictions(model, rows, size):
    model.eval()
    text = []
    for i in range(0, len(rows), size):
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            generated = model.generate(**features(rows.iloc[i:i+size]))
        text.extend(tokenizer.batch_decode(generated, skip_special_tokens=True))
    result = rows[["sample_id", "audio_name", "recording", "speaker", "speech_status",
                   "partition", "transcription"]].copy()
    result["asr_raw"] = text
    result["reference_normalized"] = result.transcription.map(normalize)
    result["asr_normalized"] = result.asr_raw.map(normalize)
    return result

def metrics(result, punctuation_to_space=False):
    refs = [normalize(s, punctuation_to_space=punctuation_to_space) for s in result.transcription]
    hyps = [normalize(s, punctuation_to_space=punctuation_to_space) for s in result.asr_raw]
    words = jiwer.process_words(refs, hyps)
    return dict(files=len(refs), empty_predictions=hyps.count(""), reference_words=words.hits +
                words.substitutions + words.deletions, substitutions=words.substitutions,
                insertions=words.insertions, deletions=words.deletions, wer=words.wer)
print(torch.cuda.get_device_name(0), "|", len(train), "train |", len(val), "ASR validation")

In [ ]:
def smoke_test():
    set_seed(SEED)
    model = make_model()
    key, param = next((n, p) for n, p in model.named_parameters() if p.requires_grad)
    before = param.detach().clone()
    opt = make_optimizer(model)
    scaler = torch.amp.GradScaler("cuda", init_scale=128)
    model.train()
    loss = loss_on(model, train.iloc[:1])
    assert torch.isfinite(loss)
    scaler.scale(loss).backward()
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(model.model.encoder.parameters(), 1.0, error_if_nonfinite=True)
    scaler.step(opt)
    scaler.update()
    assert not torch.equal(before, param), key
    sample = val.iloc[:1]
    original = predictions(model, sample, 1)
    import uuid
    temp = output_root / (".iciss_smoke_" + uuid.uuid4().hex)
    temp.mkdir()
    try:
        model.save_pretrained(temp)
        del opt, model, before, param, loss
        gc.collect()
        torch.cuda.empty_cache()
        restored = WhisperForConditionalGeneration.from_pretrained(temp).to(device)
        check = predictions(restored, sample, 1)
        assert original.asr_raw.tolist() == check.asr_raw.tolist()
    finally:
        for artifact in temp.iterdir():
            artifact.unlink()
        temp.rmdir()
    print("One update, save/reload and decoding passed.", metrics(check))
smoke_test()
gc.collect()
torch.cuda.empty_cache()

## Fine-tune and select the checkpoint

The full run starts from base weights after the smoke test. A saved training state from this exact recipe can resume at the last completed validation. When restarting a Kaggle session, restore the saved run directory (including `latest.pt`) under `/kaggle/working` before running these cells.

In [ ]:
from collections import deque

def batches_forever(n):
    # Shuffle completed epochs independently. Resume by advancing this light index iterator.
    rng = np.random.default_rng(SEED)
    while True:
        order = rng.permutation(n)
        for start in range(0, n, batch_size):
            yield order[start:start + batch_size]

set_seed(SEED)
model = make_model()
optimizer = make_optimizer(model)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=100, num_training_steps=steps)
scaler = torch.amp.GradScaler("cuda", init_scale=128)
recipe = dict(split=audit["manifest_sha256"], base_revision=BASE_REV, data_revision=DATA_REV,
              seed=SEED, steps=steps, batch=batch_size, accumulation=accumulation,
              eval_every=eval_every, optimizer="AdamW", lr=5e-6, weight_decay=0.01,
              scheduler="linear", warmup=100, clip=1.0, max_generation_length=225,
              weight_decay_exclusions=["bias", "LayerNorm"],
              metric="punctuation-deletion corpus WER", recipe_version=2,
              torch_version=str(torch.__version__), transformers_version=version("transformers"))
latest = run_dir / "latest.pt"
step, consumed, best_step, best_wer = 0, 0, 0, float("inf")
history, best_state = [], None
if latest.exists():
    state = torch.load(latest, map_location="cpu", weights_only=True)
    if state["recipe"] != recipe:
        raise ValueError("Run recipe mismatch. Use the original config or a new output root.")
    model.load_state_dict(state["model"])
    optimizer.load_state_dict(state["optimizer"])
    scheduler.load_state_dict(state["scheduler"])
    scaler.load_state_dict(state["scaler"])
    torch.set_rng_state(state["cpu_rng"])
    torch.cuda.set_rng_state(state["gpu_rng"])
    step, consumed = state["step"], state["consumed"]
    best_step, best_wer = state["best_step"], state["best_wer"]
    history, best_state = state["history"], state["best_state"]
    del state
    print("Restored optimizer step", step)
batches = batches_forever(len(train))
deque(itertools.islice(batches, consumed), maxlen=0)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Trainable encoder parameters:", trainable)
recent_losses = []
while step < steps:
    model.train()
    optimizer.zero_grad(set_to_none=True)
    losses = []
    for _ in range(accumulation):
        batch = train.iloc[next(batches)]
        consumed += 1
        loss = loss_on(model, batch)
        if not torch.isfinite(loss):
            raise RuntimeError("Non-finite training loss at step " + str(step))
        scaler.scale(loss / accumulation).backward()
        losses.append(loss.detach().item())
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.model.encoder.parameters(), 1.0)
    old_scale = scaler.get_scale()
    scaler.step(optimizer)
    scaler.update()
    if scaler.get_scale() < old_scale:
        continue
    scheduler.step()
    step += 1
    recent_losses.extend(losses)
    if step % 25 == 0:
        print(f"Step {step}/{steps} | mean loss {np.mean(recent_losses):.4f}", flush=True)
        recent_losses.clear()
    if step % eval_every == 0 or step == steps:
        score = metrics(predictions(model, val, eval_batch_size))
        full_epochs, remaining_batches = divmod(consumed, math.ceil(len(train) / batch_size))
        recordings_seen = full_epochs * len(train) + remaining_batches * batch_size
        history.append(dict(step=step, recordings_seen=recordings_seen, epochs_seen=recordings_seen / len(train), **score))
        if score["wer"] < best_wer:
            best_step, best_wer = step, score["wer"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        state = dict(recipe=recipe, step=step, consumed=consumed, best_step=best_step,
                     best_wer=best_wer, model=model.state_dict(), best_state=best_state,
                     optimizer=optimizer.state_dict(), scaler=scaler.state_dict(),
                     scheduler=scheduler.state_dict(), history=history, cpu_rng=torch.get_rng_state(),
                     gpu_rng=torch.cuda.get_rng_state())
        temporary = latest.with_suffix(".tmp")
        torch.save(state, temporary)
        temporary.replace(latest)
        del state
        print(f"Validation WER {100*score['wer']:.2f}% | best step {best_step}", flush=True)
model.load_state_dict(best_state)
model.eval()
model.save_pretrained(run_dir / "selected_checkpoint")
processor.save_pretrained(run_dir / "selected_checkpoint")
pd.DataFrame(history).to_csv(run_dir / "validation_history.csv", index=False)

## Export the selected model and reproducible predictions

Exports ASR-validation and pause-calibration outputs, including empty hypotheses and both normalization scores. Keep the same selected checkpoint for final inference after locking the pause rule.

In [ ]:
expected = predictions(model, val.iloc[:2], 2)
del optimizer, scheduler, scaler, model, best_state
gc.collect()
torch.cuda.empty_cache()
selected = WhisperForConditionalGeneration.from_pretrained(run_dir / "selected_checkpoint").to(device)
assert predictions(selected, val.iloc[:2], 2).asr_raw.tolist() == expected.asr_raw.tolist()
summary = {}
for label, frame in [("asr_validation", val), ("pause_calibration", calibration)]:
    result = predictions(selected, frame, eval_batch_size)
    assert result.audio_name.is_unique and set(result.audio_name) == set(frame.audio_name)
    result["reference_punct_to_space"] = result.transcription.map(lambda s: normalize(s, punctuation_to_space=True))
    result["asr_punct_to_space"] = result.asr_raw.map(lambda s: normalize(s, punctuation_to_space=True))
    result.to_csv(run_dir / f"{label}_predictions.csv", index=False)
    summary[label] = {"primary": metrics(result), "punct_to_space": metrics(result, True)}
best_record = next(row for row in history if row["step"] == best_step)
metadata = dict(recipe=recipe, selected_step=best_step, selection_wer=best_wer,
                selected_epochs_seen=best_record["epochs_seen"], total_epochs_seen=history[-1]["epochs_seen"],
                trainable_parameters=trainable, gpu=torch.cuda.get_device_name(0),
                validation_and_calibration=summary)
(run_dir / "training_summary.json").write_text(json.dumps(metadata, indent=2))
archive = output_root / "iciss_grouped_task2.zip"
with zipfile.ZipFile(archive.with_suffix(".tmp"), "w", compression=zipfile.ZIP_DEFLATED, compresslevel=1) as z:
    for path in sorted(run_dir.rglob("*")):
        if path.is_file() and path.name not in ("latest.pt", "latest.tmp"):
            z.write(path, path.relative_to(run_dir))
archive.with_suffix(".tmp").replace(archive)
print("Selected step:", best_step, "| selected validation WER:", best_wer)
print("Download:", archive)